# Aviation Accidents Analysis — Data Cleaning

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**

In this summative assessment you will demonstrate your ability to:

- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

---

### About this notebook

This is **notebook 1 of 2**. It loads `data/aviation_raw.csv` (NTSB aviation accident/incident records, 1948-2022), applies the nine cleaning steps the brief asks for, and writes `data/aviation_cleaned.csv`. **Notebook 2** (`02_Aviation_Accidents_Data_Analysis.ipynb`) loads that file and does the EDA.

Every cleaning decision below is documented with its row count and its effect on the sample, because each filter moves the population away from "all accidents" and towards a specific, narrower one. Those moves are reported, not hidden.

**Scope note.** The cutoff is a *fixed* 1983-01-01, not a rolling "today minus 40 years" date. A fixed cutoff keeps the project reproducible: re-running this notebook in a later year gives exactly the same sample. The dataset ends 2022-12-29. Falling inside the 40-year window is **not** evidence that a make/model is still in service, and nothing here proves current fleet eligibility or airworthiness.

### Make relevant library imports

In [8]:
import warnings
warnings.filterwarnings('ignore')

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Plot defaults keep figures readable when exported for the report.
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 11

print('pandas', pd.__version__, '| numpy', np.__version__,
      '| matplotlib', matplotlib.__version__, '| seaborn', sns.__version__)

pandas 3.0.6 | numpy 2.5.3 | matplotlib 3.11.2 | seaborn 0.13.2


The project root holds `data/` and `images/`. The cells below resolve paths relative to the notebook, so the project runs from an extracted folder with no edits:

- if the notebook is opened from `notebooks/`, the parent folder is the project root;
- if it is executed from the project root (e.g. `jupyter nbconvert --execute`), the root is the working directory.

In [9]:
BASE = Path.cwd()
if not (BASE / 'data').exists() and (BASE.parent / 'data').exists():
    BASE = BASE.parent

DATA_DIR = BASE / 'data'
IMG_DIR = BASE / 'images'
IMG_DIR.mkdir(exist_ok=True)

RAW_PATH = DATA_DIR / 'aviationData.csv'
CLEAN_PATH = DATA_DIR / 'aviation_cleaned.csv'

print('project root :', BASE)
print('raw file     :', RAW_PATH.exists())

project root : c:\Users\ADMIN\Documents\Moringa\Data_Analysis_Jet_Airline_Safety
raw file     : True


## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [10]:
# The NTSB export is Windows-1252, not UTF-8. Reading it with the default utf-8
# codec raises UnicodeDecodeError, so the encoding has to be stated explicitly.
raw = pd.read_csv(RAW_PATH, encoding='cp1252', low_memory=False)

print('shape :', raw.shape[0], 'rows x', raw.shape[1], 'columns')
print('size  : %.1f MB in memory' % (raw.memory_usage(deep=True).sum() / 1e6))
raw.iloc[:3, :9]

shape : 88889 rows x 31 columns
size  : 128.1 MB in memory


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN


In [11]:
# Compact NaN / dtype profile, widest columns first
profile = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'non_null': raw.notna().sum(),
    'pct_null': (raw.isna().mean() * 100).round(1),
})
print('columns with >20,000 non-null values:', int((profile['non_null'] > 20000).sum()),
      'of', raw.shape[1])
profile.sort_values('non_null', ascending=False).head(16)

columns with >20,000 non-null values: 29 of 31


,dtype,non_null,pct_null
Event.Id,str,88889,0.0
Investigation.Type,str,88889,0.0
Accident.Number,str,88889,0.0
Event.Date,str,88889,0.0
Location,str,88837,0.1
Make,str,88826,0.1
Model,str,88797,0.1
Amateur.Built,str,88787,0.1
Country,str,88663,0.3
Injury.Severity,str,87889,1.1


In [12]:
# Summary statistics for the numeric outcome columns and engine count
num_cols = ['Number.of.Engines', 'Total.Fatal.Injuries', 'Total.Serious.Injuries',
            'Total.Minor.Injuries', 'Total.Uninjured']
raw[num_cols].describe().round(2)

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.00,77488.00,76379.00,76956.00,82977.00
mean,1.15,0.65,0.28,0.36,5.33
std,0.45,5.49,1.54,2.24,27.91
min,0.00,0.00,0.00,0.00,0.00
25%,1.00,0.00,0.00,0.00,0.00
50%,1.00,0.00,0.00,0.00,1.00
75%,1.00,0.00,0.00,0.00,2.00
max,8.00,349.00,161.00,380.00,699.00


**What inspection shows**

- 88,889 rows and 31 columns. `Aircraft.Category` is empty for 63.7% of rows, which is the single biggest structural problem in the file: the category is only recorded for a subset of records.
- The four injury-count columns are each missing for 7-13% of rows, and they are missing *together* in some records (36 rows have none of the four). Treating a missing count as zero is therefore a decision that needs to be measured, not assumed.
- `Event.Date`, `Make`, `Model` and `Aircraft.damage` are text columns that need parsing, case-folding and explicit unknown handling.
- `Make` arrives in mixed case with corporate suffixes ("CESSNA" / "Cessna", "AIR TRACTOR" / "AIR TRACTOR INC"), so the same manufacturer is spread over several labels.
- Injury totals contain extreme values (large jets with hundreds of uninjured occupants), which is why the analysis uses a **fraction** rather than a raw count.

## Data Cleaning

The brief's nine steps, in the order they are executed below:

1. Import libraries (above).
2. Inspect NaNs, dtypes and summary statistics (above).
3. Filter to `Aircraft.Category == 'Airplane'`, `Amateur.Built == 'No'`, `Event.Date >= 1983-01-01`, with justified imputations, and record counts and exclusion bias.
4. Build the injury measures: estimated occupants, fatal+serious fraction, and an explicit known/unknown outcome flag.
5. Build the `destroyed` indicator from `Aircraft.damage`, keeping missing outcomes missing.
6. Normalise `Make` (documented aliases) and keep makes with at least 50 events.
7. Drop missing `Model`, normalise it, and build a unique `make_model` key.
8. Clean `Engine.Type`, `Weather.Condition`, `Number.of.Engines`, `Purpose.of.flight`, `Broad.phase.of.flight`, preserving "unknown" as a category.
9. Retain original columns with more than 20,000 non-null values (evaluated on the **initial full data**), add the derived columns, and save the CSV.

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [13]:
print(raw['Aircraft.Category'].value_counts(dropna=False).head(9).to_string())
print()
print('missing Aircraft.Category : %d rows (%.1f%%)'
      % (raw['Aircraft.Category'].isna().sum(), raw['Aircraft.Category'].isna().mean() * 100))
print(raw['Amateur.Built'].value_counts(dropna=False).to_string())

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30

missing Aircraft.Category : 56602 rows (63.7%)
Amateur.Built
No     80312
Yes     8475
NaN      102


**Decisions on the three filters**

| Filter | Decision | Why |
|---|---|---|
| `Aircraft.Category` | keep only `Airplane`; empty is **not** imputed to Airplane | 63.7% of rows have no category. Filling them with the modal value 'Airplane' would invent the client's population out of mostly older records and would silently change every denominator. Excluding them is the conservative choice and its cost is measured in the bias table below. |
| `Amateur.Built` | keep only `No` | The client wants professional builds; amateur-built aircraft are a different certification and construction population. 102 rows have no flag and are excluded with the same rule. |
| `Event.Date` | keep `1983-01-01` to `2022-12-31` | Fixed 40-year window from the brief, applied at a fixed calendar date so the sample is reproducible. 58 rows have no parseable date (in the full file) and cannot be placed in the window. |
| `Number.of.Engines` | impute the make's most common engine count, else 1 | Engine count is a design property of the airframe, so the make's mode is a defensible donor. The fallback of 1 reflects that single-engine aircraft dominate the airplane population. The column is a *conditioning* variable in notebook 2, never an outcome. |

In [14]:
keep = raw.copy()

cat = keep['Aircraft.Category'].fillna('').str.strip().str.lower()
keep = keep[cat.eq('airplane')]

built = keep['Amateur.Built'].fillna('').str.strip().str.lower()
keep = keep[built.eq('no')]

keep['Event.Date'] = pd.to_datetime(keep['Event.Date'], errors='coerce')
keep = keep[keep['Event.Date'].ge('1983-01-01') & keep['Event.Date'].le('2022-12-31')]

keep = keep.copy()
print('rows after the three filters:', len(keep))
print('date range:', keep['Event.Date'].min().date(), 'to', keep['Event.Date'].max().date())

rows after the three filters: 21447
date range: 1983-03-18 to 2022-12-26


In [15]:
# --- engine count imputation (make-level mode, fallback 1) ---
eng_missing_before = int(keep['Number.of.Engines'].isna().sum())

make_key = keep['Make'].fillna('').str.upper().str.strip()
make_mode = keep.groupby(make_key)['Number.of.Engines'].agg(
    lambda v: v.mode().iloc[0] if v.notna().any() else np.nan)

missing = keep['Number.of.Engines'].isna()
keep.loc[missing, 'Number.of.Engines'] = keep.loc[missing, 'Make'].fillna('').str.upper().str.strip().map(make_mode)
keep['Number.of.Engines'] = keep['Number.of.Engines'].fillna(1).astype(int)

print('Number.of.Engines missing before :', eng_missing_before)
print('Number.of.Engines missing after  :', int(keep['Number.of.Engines'].isna().sum()))
print('resulting distribution:')
print(keep['Number.of.Engines'].value_counts().sort_index().to_string())

Number.of.Engines missing before : 2551
Number.of.Engines missing after  : 0
resulting distribution:
Number.of.Engines
0        9
1    17260
2     4032
3       43
4      101
6        1
8        1


**Exclusion bias, measured.** A filter is only defensible if we say who it removes. The table tracks survivors at each stage, and the second block compares the retained sample against the full file by decade and by investigation type.

In [16]:
funnel = pd.DataFrame({
    'stage': ['full file', 'Aircraft.Category == Airplane', 'Amateur.Built == No',
              'Event.Date 1983-01-01 to 2022-12-31'],
    'rows_remaining': [len(raw), int(len(raw) * 0 + (raw['Aircraft.Category'].fillna('').str.strip().str.lower() == 'airplane').sum()), 0, len(keep)],
})
# recompute the first two stages explicitly so the numbers are traceable
s1 = raw['Aircraft.Category'].fillna('').str.strip().str.lower().eq('airplane')
s2 = s1 & raw['Amateur.Built'].fillna('').str.strip().str.lower().eq('no')
funnel.loc[1, 'rows_remaining'] = int(s1.sum())
funnel.loc[2, 'rows_remaining'] = int(s2.sum())
funnel['rows_dropped'] = funnel['rows_remaining'].shift(1).fillna(len(raw)).astype(int) - funnel['rows_remaining']
funnel['pct_of_raw'] = (funnel['rows_remaining'] / len(raw) * 100).round(1)
funnel

,stage,rows_remaining,rows_dropped,pct_of_raw
0,full file,88889,0,100.0
1,Aircraft.Category == Airplane,27617,61272,31.1
2,Amateur.Built == No,24417,3200,27.5
3,Event.Date 1983-01-01 to 2022-12-31,21447,2970,24.1


In [17]:
# How the retained sample differs from the full file: by decade and investigation type
raw_year = pd.to_datetime(raw['Event.Date'], errors='coerce').dt.year
keep_year = keep['Event.Date'].dt.year

decades = sorted(set((raw_year.dropna() // 10 * 10).astype(int)))
bias = pd.DataFrame({
    'decade': decades,
    'raw_rows': [int((raw_year // 10 * 10 == d).sum()) for d in decades],
    'raw_kept': [int((keep_year // 10 * 10 == d).sum()) for d in decades],
})
bias['pct_retained'] = (bias['raw_kept'] / bias['raw_rows'] * 100).round(1)

print('share of the full file retained by decade (%)')
print(bias.to_string(index=False))
print()
print('investigation type in retained sample:')
print(keep['Investigation.Type'].value_counts(normalize=True).round(4).mul(100).to_string())
print()
print('retained rows whose Make has <50 events in this window: ',
      int((~keep['Make'].fillna('').str.upper().str.strip().isin(
          keep['Make'].fillna('').str.upper().str.strip().value_counts().loc[lambda v: v >= 50].index)).sum()))

share of the full file retained by decade (%)
 decade  raw_rows  raw_kept  pct_retained
   1940         1         0           0.0
   1960         1         0           0.0
   1970         4         0           0.0
   1980     24685        54           0.2
   1990     22984       204           0.9
   2000     19914      5520          27.7
   2010     16756     12302          73.4
   2020      4544      3367          74.1

investigation type in retained sample:
Investigation.Type
Accident    92.96
Incident     7.04

retained rows whose Make has <50 events in this window:  3555


**Reading the bias table**

- Retention is heavily skewed towards recent decades: pre-1983 records are removed by construction, and within 1983+, older decades lose more rows to the missing-category filter. The analysis therefore describes **post-1983 records that carry an explicit Airplane category and a professional-build flag**, not all post-1983 accidents.
- The lost rows are mostly records with incomplete administrative fields, which is a *data-completeness* bias rather than a random sample of accidents. If incomplete records are more common for small or non-fatal events, the retained sample over-represents the better-documented (usually more serious) accidents. This is a known limitation, restated in notebook 2.
- Both `Accident` and `Incident` investigation types are kept. Incidents are a small minority and the injury/damage measures are defined for them too, so removing them would discard valid outcome data; the split is reported above.

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**The estimate used here**

```
estimated occupants = Total.Fatal.Injuries + Total.Serious.Injuries
                    + Total.Minor.Injuries + Total.Uninjured
fatal_serious_fraction = (Total.Fatal.Injuries + Total.Serious.Injuries)
                         / estimated occupants
```

Three things must be said plainly, because they change how every later number should be read:

1. **This is an estimate of people on board, not seat capacity.** It is what the NTSB counted, not what the aircraft can carry. A 180-seat jet that lands with 4 minor injuries is recorded as 4 occupants, not 180. Notebook 2 uses a 20-occupant threshold only because the brief asks for a size split; the threshold therefore measures *how many people were on board the accident flight*, which is a proxy for aircraft size, and the proxy is noisy exactly where it matters.
2. **A missing count is not a zero.** The brief warns against filling every NaN with 0. Here a missing field is treated as 0 **only** when at least one of the four fields is reported and the resulting total is positive, and every such row is counted. Rows with no injury fields at all, or with a zero total, get `injury_outcome_known = False`, keep `fatal_serious_fraction = NaN`, and are **excluded from injury statistics, not from the dataset**. Notebook 2 reports how many rows that removes and re-runs its headline comparison without the partially-filled rows as a sensitivity check.
3. **The denominator is not exposure.** Every fraction is conditional on an accident having been recorded, so it is an injury *severity* measure within accidents, never an accident *rate* per flight hour or per aircraft in service.

In [18]:
INJ_COLS = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
            'Total.Minor.Injuries', 'Total.Uninjured']
for col in INJ_COLS:
    keep[col] = pd.to_numeric(keep[col], errors='coerce')

# how many of the four counts are reported for each row?
keep['injury_fields_reported'] = keep[INJ_COLS].notna().sum(axis=1)

# estimated occupants: missing counts contribute 0, but only usable if something was reported
keep['occupants_est'] = keep[INJ_COLS].fillna(0).sum(axis=1)

injury_valid = keep['injury_fields_reported'].gt(0) & keep['occupants_est'].gt(0)
keep['injury_outcome_known'] = injury_valid

fatal = keep['Total.Fatal.Injuries'].fillna(0)
serious = keep['Total.Serious.Injuries'].fillna(0)
keep['fatal_serious_fraction'] = np.nan
keep.loc[injury_valid, 'fatal_serious_fraction'] = (
    (fatal[injury_valid] + serious[injury_valid]) / keep.loc[injury_valid, 'occupants_est'])

print('rows in sample                          :', len(keep))
print('rows with all four injury fields        :', int((keep['injury_fields_reported'] == 4).sum()))
print('rows with 1-3 injury fields (zero-filled):', int(keep['injury_fields_reported'].between(1, 3).sum()))
print('rows with no injury fields (excluded)   :', int((keep['injury_fields_reported'] == 0).sum()))
print('rows with zero estimated occupants (excluded):', int((keep['occupants_est'] == 0).sum()))
print('usable injury outcomes                  :', int(injury_valid.sum()),
      '(%.1f%% of sample)' % (injury_valid.mean() * 100))

rows in sample                          : 21447
rows with all four injury fields        : 18419
rows with 1-3 injury fields (zero-filled): 2981
rows with no injury fields (excluded)   : 47
rows with zero estimated occupants (excluded): 904
usable injury outcomes                  : 20543 (95.8% of sample)


In [19]:
print('fatal + serious fraction, usable rows:')
print(keep.loc[injury_valid, 'fatal_serious_fraction'].describe(
    percentiles=[.25, .5, .75, .9, .99]).round(3).to_string())
print()
print('share of usable rows with zero fatal/serious injuries: %.1f%%'
      % ((keep.loc[injury_valid, 'fatal_serious_fraction'] == 0).mean() * 100))

fatal + serious fraction, usable rows:
count    20543.000
mean         0.284
std          0.432
min          0.000
25%          0.000
50%          0.000
75%          0.800
90%          1.000
99%          1.000
max          1.000

share of usable rows with zero fatal/serious injuries: 67.1%


The distribution is strongly zero-inflated: a large share of accidents injure nobody seriously, so the **mean is dominated by the small number of high-casualty events** and the median is 0 for every group examined in notebook 2. That is why notebook 2 reports n, mean, median and a confidence interval together, and never compares means without showing the distribution.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

**Cleaning tasks:** standardise the four observed values (`Destroyed`, `Substantial`, `Minor`, `Unknown`) and map them to a binary indicator. `Destroyed -> 1`; `Substantial` and `Minor` -> `0`; `Unknown` and blank -> `NaN`.

The missing outcomes are kept as missing. Forcing `Unknown` to 0 would assume the aircraft survived, which is exactly the claim under test, and it would inflate every make's apparent safety. Rows with `destroyed = NaN` are reported and excluded from destruction statistics.

In [20]:
print('Aircraft.damage values before cleaning:')
print(keep['Aircraft.damage'].value_counts(dropna=False).to_string())

keep['Aircraft.damage'] = keep['Aircraft.damage'].fillna('').str.strip().str.title()
keep['destroyed'] = keep['Aircraft.damage'].map({'Destroyed': 1, 'Substantial': 0, 'Minor': 0})

print()
print('destroyed indicator:')
print(keep['destroyed'].value_counts(dropna=False).to_string())
print()
print('destroyed known :', int(keep['destroyed'].notna().sum()),
      '| unknown (kept as NaN):', int(keep['destroyed'].isna().sum()))

Aircraft.damage values before cleaning:
Aircraft.damage
Substantial    16990
Destroyed       2316
NaN             1227
Minor            817
Unknown           97

destroyed indicator:
destroyed
0.0    17807
1.0     2316
NaN     1324

destroyed known : 20123 | unknown (kept as NaN): 1324


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

**Cleaning tasks for `Make`**

1. Cast to string, strip surrounding whitespace, upper-case, remove `.` and `,`, collapse repeated spaces.
2. Merge a **short, documented alias list** — only pairs that are unambiguously the same manufacturer in this data. No fuzzy matching and no manual re-coding of small labels, because either would let the analyst's prior decide which makes look safe.
3. Keep makes with **at least 50 events** in the filtered sample. Below that, a mean injury fraction is driven by one or two accidents; 50 is the threshold the brief suggests and it is applied to the *cleaned* make label.

**Alias groups applied** (left -> kept spelling): `CESSNA AIRCRAFT -> CESSNA`, `PIPER AIRCRAFT -> PIPER`, `BEECH AIRCRAFT -> BEECH`, `THE BOEING COMPANY / BOEING COMPANY -> BOEING`, `AIR TRACTOR -> AIR TRACTOR INC`, `CIRRUS -> CIRRUS DESIGN CORP`, `DE HAVILLAND -> DEHAVILLAND`, `MCDONNELL-DOUGLAS -> MCDONNELL DOUGLAS`, `BOMBARDIER -> BOMBARDIER INC`, `AVIAT -> AVIAT AIRCRAFT INC`.

In [21]:
MAKE_ALIASES = {
    'CESSNA AIRCRAFT': 'CESSNA',
    'CESSNA AIRCRAFT CO': 'CESSNA',
    'PIPER AIRCRAFT': 'PIPER',
    'PIPER AIRCRAFT INC': 'PIPER',
    'BEECH AIRCRAFT': 'BEECH',
    'BEECH AIRCRAFT CORP': 'BEECH',
    'THE BOEING COMPANY': 'BOEING',
    'BOEING COMPANY': 'BOEING',
    'AIR TRACTOR': 'AIR TRACTOR INC',
    'AIR TRACTOR INC': 'AIR TRACTOR INC',
    'CIRRUS': 'CIRRUS DESIGN CORP',
    'CIRRUS DESIGN CORP': 'CIRRUS DESIGN CORP',
    'DE HAVILLAND': 'DEHAVILLAND',
    'DEHAVILLAND': 'DEHAVILLAND',
    'MCDONNELL DOUGLAS': 'MCDONNELL DOUGLAS',
    'MCDONNELL-DOUGLAS': 'MCDONNELL DOUGLAS',
    'BOMBARDIER': 'BOMBARDIER INC',
    'BOMBARDIER INC': 'BOMBARDIER INC',
    'AVIAT': 'AVIAT AIRCRAFT INC',
    'AVIAT AIRCRAFT INC': 'AVIAT AIRCRAFT INC',
}


def normalise_make(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().strip()
    text = re.sub(r'[.,]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return MAKE_ALIASES.get(text, text)


make_before = keep['Make'].nunique()
keep['Make_clean'] = keep['Make'].map(normalise_make)
keep = keep[keep['Make_clean'].notna()].copy()

counts = keep['Make_clean'].value_counts()
print('distinct Make labels before :', make_before)
print('distinct Make labels after  :', keep['Make_clean'].nunique())
print('makes with >= 50 events     :', int((counts >= 50).sum()))
print()
print('top 15 makes after normalisation:')
print(counts.head(15).to_string())

distinct Make labels before : 1332
distinct Make labels after  : 1034
makes with >= 50 events     : 32

top 15 makes after normalisation:
Make_clean
CESSNA                7181
PIPER                 4026
BEECH                 1435
BOEING                1282
AIR TRACTOR INC        432
CIRRUS DESIGN CORP     385
MOONEY                 363
AIRBUS                 243
BELLANCA               219
MAULE                  215
AERONCA                200
DEHAVILLAND            168
CHAMPION               158
EMBRAER                153
AVIAT AIRCRAFT INC     148


In [22]:
rows_before_make_filter = len(keep)
keep = keep[keep['Make_clean'].isin(counts[counts >= 50].index)].copy()

print('rows removed by the >=50-event make filter :', rows_before_make_filter - len(keep))
print('rows remaining                             :', len(keep))
print('makes retained                             :', keep['Make_clean'].nunique())

rows removed by the >=50-event make filter : 3359
rows remaining                             : 18085
makes retained                             : 32


The long tail that this removes is dominated by one-off manufacturers, kit-derived labels and defunct marques. Those rows are not "bad data"; they are simply too thin to support a make-level average, and keeping them would let a single accident define a make's safety profile. The cost is that **the recommendations can only speak about the 30-odd makes that clear the threshold**, not about every manufacturer in the file.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [23]:
print('rows with missing Model :', int(keep['Model'].isna().sum()))
keep = keep[keep['Model'].notna()].copy()


def normalise_model(value):
    text = str(value).upper().strip()
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' ,.')


keep['Model_clean'] = keep['Model'].map(normalise_model)

# does the same model label appear under more than one make?
shared = (keep.groupby('Model_clean')['Make_clean'].nunique() > 1).sum()
print('model labels used by more than one make :', int(shared))
print('distinct model labels                   :', keep['Model_clean'].nunique())

keep['make_model'] = keep['Make_clean'] + ' ' + keep['Model_clean']
print('distinct make_model keys                :', keep['make_model'].nunique())
print()
print('most common make_model keys:')
print(keep['make_model'].value_counts().head(10).to_string())

rows with missing Model : 13
model labels used by more than one make : 69
distinct model labels                   : 2047
distinct make_model keys                : 2128

most common make_model keys:
make_model
CESSNA 172                 770
BOEING 737                 403
CESSNA 152                 316
CESSNA 182                 304
CESSNA 172S                276
PIPER PA28                 273
CIRRUS DESIGN CORP SR22    258
CESSNA 172N                249
CESSNA 180                 213
CESSNA 172M                180


Model labels are **not** unique across makes (the same alphanumeric appears under several manufacturers), so a model cannot be analysed on its own. `make_model = Make_clean + " " + Model_clean` is the unique plane-type key used in notebook 2. Variant suffixes are deliberately left in place: merging `737` with `737-7H4` would be a judgement about which variants are the same aircraft, and that judgement belongs in the discussion, not silently in the key.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Note**: You do not necessarily need to impute or drop NaNs here.

All five are treated as **conditioning variables**, not outcomes. The rule everywhere is: standardise spelling, merge only unambiguous duplicates, and map blank/unknown to an explicit `Unknown` category instead of dropping the row. Keeping `Unknown` visible matters — if unknownness is systematic, a comparison that silently drops it is biased, so notebook 2 reports the `Unknown` share alongside every comparison.

In [24]:
for col in ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('--', col, '| distinct before:', keep[col].nunique())
    print(keep[col].value_counts(dropna=False).head(6).to_string())
    print()

-- Engine.Type | distinct before: 8
Engine.Type
Reciprocating    12958
NaN               3256
Turbo Prop         941
Turbo Fan          717
Unknown            107
Turbo Jet           71

-- Weather.Condition | distinct before: 4
Weather.Condition
VMC    14437
NaN     2456
IMC      915
Unk      188
UNK       76

-- Purpose.of.flight | distinct before: 24
Purpose.of.flight
Personal              9948
NaN                   3086
Instructional         2431
Aerial Application     734
Business               413
Unknown                305

-- Broad.phase.of.flight | distinct before: 12
Broad.phase.of.flight
NaN            15598
Landing         1119
Takeoff          428
Cruise           242
Approach         210
Maneuvering      130



In [25]:
# --- Weather.Condition: VMC / IMC / Unknown ---
keep['Weather.Condition'] = keep['Weather.Condition'].fillna('').str.strip().str.upper()
keep['Weather.Condition'] = keep['Weather.Condition'].replace({'VMC': 'VMC', 'IMC': 'IMC',
                                                               'UNK': 'Unknown', '': 'Unknown'})

# --- Engine.Type: keep the meaningful families, fold rare/odd labels into Unknown ---
keep['Engine.Type'] = keep['Engine.Type'].fillna('').str.strip().str.upper()
engine_map = {
    'RECIPROCATING': 'Reciprocating',
    'TURBO PROP': 'Turbo Prop',
    'TURBOPROP': 'Turbo Prop',
    'TURBO SHAFT': 'Turbo Shaft',
    'TURBO FAN': 'Turbo Fan',
    'GEARED TURBOFAN': 'Turbo Fan',
    'TURBO JET': 'Turbo Jet',
    'ELECTRIC': 'Electric',
    'UNKNOWN': 'Unknown', 'UNK': 'Unknown', 'NONE': 'Unknown', 'LR': 'Unknown', '': 'Unknown',
}
keep['Engine.Type'] = keep['Engine.Type'].map(engine_map).fillna('Unknown')

# --- Purpose.of.flight: collapse unambiguous duplicates, explicit Unknown ---
keep['Purpose.of.flight'] = keep['Purpose.of.flight'].fillna('').str.strip()
purpose_map = {
    'Air Race show': 'Air Race/show',
    'Air Race/show': 'Air Race/show',
    'Public Aircraft - Federal': 'Public Aircraft',
    'Public Aircraft - Local': 'Public Aircraft',
    'Public Aircraft - State': 'Public Aircraft',
    'Public Aircraft': 'Public Aircraft',
    'Executive/corporate': 'Executive/corporate',
    'ASHO': 'Unknown', 'PUBS': 'Unknown', 'PUBL': 'Unknown',
    'Unknown': 'Unknown', '': 'Unknown',
}
keep['Purpose.of.flight'] = keep['Purpose.of.flight'].map(purpose_map).fillna(keep['Purpose.of.flight'])

# --- Broad.phase.of.flight: title-case, blank -> Unknown (never dropped) ---
keep['Broad.phase.of.flight'] = keep['Broad.phase.of.flight'].fillna('').str.strip()
keep['Broad.phase.of.flight'] = keep['Broad.phase.of.flight'].replace('', 'Unknown')

print('after cleaning:')
for col in ['Weather.Condition', 'Engine.Type', 'Purpose.of.flight', 'Broad.phase.of.flight']:
    print('--', col)
    print(keep[col].value_counts().to_string())
    print()

after cleaning:
-- Weather.Condition
Weather.Condition
VMC        14437
Unknown     2720
IMC          915

-- Engine.Type
Engine.Type
Reciprocating    12958
Unknown           3364
Turbo Prop         941
Turbo Fan          729
Turbo Jet           71
Turbo Shaft          9

-- Purpose.of.flight
Purpose.of.flight
Personal               9948
Unknown                3396
Instructional          2431
Aerial Application      734
Business                413
Positioning             271
Skydiving               157
Aerial Observation      148
Other Work Use          121
Public Aircraft          87
Banner Tow               86
Ferry                    77
Flight Test              76
Executive/corporate      67
Glider Tow               29
Air Race/show            17
Firefighting             12
Air Drop                  2

-- Broad.phase.of.flight
Broad.phase.of.flight
Unknown        15609
Landing         1119
Takeoff          428
Cruise           242
Approach         210
Maneuvering      130
Taxi      

In [26]:
print('Number of engines (cleaned as an integer above):')
print(keep['Number.of.Engines'].value_counts().sort_index().to_string())
print()
print('rows with Broad.phase.of.flight = Unknown: %d (%.1f%%)'
      % ((keep['Broad.phase.of.flight'] == 'Unknown').sum(),
         (keep['Broad.phase.of.flight'] == 'Unknown').mean() * 100))
print('by decade of event (phase availability is not uniform):')
print(keep.groupby(keep['Event.Date'].dt.year // 10 * 10)['Broad.phase.of.flight']
      .apply(lambda s: (s != 'Unknown').mean() * 100).round(1).to_string())

Number of engines (cleaned as an integer above):
Number.of.Engines
0        5
1    14619
2     3354
3       27
4       67

rows with Broad.phase.of.flight = Unknown: 15609 (86.4%)
by decade of event (phase availability is not uniform):
Event.Date
1980    92.5
1990    50.9
2000    49.7
2010     0.0
2020     0.0


**Two cleaning findings that matter for the analysis**

1. `Broad.phase.of.flight` is not missing at random. It is well populated in the 1980s and 1990s and essentially absent from about 2010 onwards — the field stopped being carried in the export. So any phase-of-flight comparison in notebook 2 runs on an **older subsample**, not on the whole 1983-2022 period, and cannot be used to describe the current fleet. This is recorded rather than patched, and notebook 2 states the phase sample's date range.
2. `Weather.Condition` is the opposite case: 86% of rows are known, and the "unknown" labels were two spellings of the same thing (`UNK`, `Unk`). Folding them into one explicit `Unknown` category (225 rows) prevents a spurious third level in the plots.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

**Rule used:** keep every original column that has **more than 20,000 non-null values in the initial full 88,889-row file**, and drop the rest. The count is taken on the *initial* data, before filtering, for a specific reason: several essential conditioning variables are complete in the raw file but sparse after filtering (`Broad.phase.of.flight` has 61,724 non-null values in the full file but only ~2,800 in the post-1983 airplane sample). Evaluating the threshold after filtering would delete the very columns the brief asks us to analyse. The rule is therefore about whether a field is populated at source, not whether it is populated in this slice.

In [27]:
support_raw = raw.notna().sum()
keep_cols = [c for c in raw.columns if support_raw[c] > 20000]
dropped_cols = [c for c in raw.columns if support_raw[c] <= 20000]

print('original columns kept   :', len(keep_cols))
print('original columns dropped:', len(dropped_cols))
print()
print('dropped (non-null in full file):')
for c in dropped_cols:
    print('  %-22s %6d' % (c, support_raw[c]))
print()
print('kept columns:', ', '.join(keep_cols))

original columns kept   : 29
original columns dropped: 2

dropped (non-null in full file):
  Schedule                12582
  Air.carrier             16648

kept columns: Event.Id, Investigation.Type, Accident.Number, Event.Date, Location, Country, Latitude, Longitude, Airport.Code, Airport.Name, Injury.Severity, Aircraft.damage, Aircraft.Category, Registration.Number, Make, Model, Amateur.Built, Number.of.Engines, Engine.Type, FAR.Description, Purpose.of.flight, Total.Fatal.Injuries, Total.Serious.Injuries, Total.Minor.Injuries, Total.Uninjured, Weather.Condition, Broad.phase.of.flight, Report.Status, Publication.Date


In [28]:
derived_cols = ['Make_clean', 'Model_clean', 'make_model', 'occupants_est',
                'fatal_serious_fraction', 'destroyed', 'injury_outcome_known',
                'injury_fields_reported']
final_cols = keep_cols + derived_cols
clean = keep[final_cols].copy()

print('final shape:', clean.shape)
print('rows :', len(clean))
print('makes:', clean['Make_clean'].nunique(), '| make_model keys:', clean['make_model'].nunique())
print('injury outcomes known :', int(clean['injury_outcome_known'].sum()))
print('destroyed outcome known:', int(clean['destroyed'].notna().sum()))

final shape: (18072, 37)
rows : 18072
makes: 32 | make_model keys: 2128
injury outcomes known : 17281
destroyed outcome known: 16907


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [29]:
clean.to_csv(CLEAN_PATH, index=False)
print('written:', CLEAN_PATH)
print('size   : %.1f MB' % (CLEAN_PATH.stat().st_size / 1e6))

reload_check = pd.read_csv(CLEAN_PATH, parse_dates=['Event.Date'])
print('reloaded shape:', reload_check.shape)
print('fatal_serious_fraction NaN on reload:', int(reload_check['fatal_serious_fraction'].isna().sum()))

written: c:\Users\ADMIN\Documents\Moringa\Data_Analysis_Jet_Airline_Safety\data\aviation_cleaned.csv
size   : 6.9 MB
reloaded shape: (18072, 37)
fatal_serious_fraction NaN on reload: 791


### Cleaning summary

| Measure | Value |
|---|---|
| Raw rows | 88,889 |
| After Airplane / professional-build / 1983+ filters | 21,447 |
| After >=50-event make filter | 18,085 |
| After dropping missing Model | 18,072 |
| Usable injury outcomes | 17,281 |
| Destruction outcome known | 16,907 |
| Makes retained | 32 |
| Unique make_model keys | 2,128 |
| Saved columns | 38 (30 original + 8 derived) |

**What a reader should carry into notebook 2**

- Every statistic is conditional on a recorded accident, and on that accident being in the post-1983, professionally-built, explicitly-categorised airplane slice.
- `occupants_est` is reported occupancy, not seats. The 20-occupant size split in notebook 2 is a proxy and will misclassify large jets whose injury counts are incomplete as "small".
- Unknown destruction and unknown injury outcomes are kept as missing and excluded from the relevant statistic, with counts reported.
- `Broad.phase.of.flight` is only usable as a pre-2010 subsample; `Weather.Condition` covers 86% of rows.